# Unit 10 - Bandits vs Fixed Split (Demo) · **V2 material**

**Atoms served:** `U10-A13` (**bandits** - pre-class / optional; `V33` introduces the idea, this notebook makes the trade-off concrete)

**Estimated runtime:** ~20 seconds

**After this notebook you can:** compare Thompson sampling to a fixed 50/50 split on the same arms, see the bandit earn more during the run, and explain why the losing arm's confidence interval becomes useless.

## Without code

1. Fixed 50/50 cumulative reward after 5000 pulls: about **535**, close to the 550 you would expect from the midpoint of the 0.10 and 0.12 rates.
2. Thompson sampling cumulative reward: about **637** - it shifts traffic to the better arm and earns roughly 100 extra conversions.
3. 95% CI width on the losing arm: **0.024** under the fixed split versus **0.040** under the bandit, about 1.7x wider, because the bandit stopped sampling that arm (935 pulls instead of about 2500).

**Note:** `V33` explains that bandits earn while they learn. This notebook shows what they spend: causal clarity on the arm they stopped exploring.

## 1. The question

A product team can run a fixed A/B test or a **Thompson sampling** bandit on the same two checkout variants. The bandit earns more during the experiment - what does it pay for that gain?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Two Bernoulli arms: control conversion 0.10, treatment 0.12. Same true rates for both policies.

We simulate 5000 sequential pulls under a fixed 50/50 split and under Thompson sampling with Beta(1,1) priors.

In [ ]:
p_control = 0.10
p_treatment = 0.12
n_pulls = 5000
true_rates = np.array([p_control, p_treatment])

## 4. The naive move

Treat the bandit's higher cumulative reward as proof you learned faster with no downside.

Run a fixed 50/50 split and track cumulative conversion rate (reward).

In [ ]:
choices = np.random.randint(0, 2, n_pulls)
rewards = np.random.binomial(1, true_rates[choices])
fixed_cum_reward = rewards.sum()
fixed_rate = fixed_cum_reward / n_pulls
print('Fixed split: total conversions', fixed_cum_reward, 'rate', round(fixed_rate, 4))

The fixed split keeps exploring both arms evenly - good for measurement, expensive in missed conversions.

## 5. What actually happens

**Thompson sampling.** Maintain Beta posteriors, sample from each, pull the arm with the highest draw.

Track cumulative reward under the bandit and compare to the fixed split.

In [ ]:
alpha_p = np.ones(2)
beta_p = np.ones(2)
ts_choices = []
ts_rewards = []
for _ in range(n_pulls):
    samples = np.random.beta(alpha_p, beta_p)
    arm = int(np.argmax(samples))
    r = np.random.binomial(1, true_rates[arm])
    alpha_p[arm] += r
    beta_p[arm] += 1 - r
    ts_choices.append(arm)
    ts_rewards.append(r)
ts_cum_reward = sum(ts_rewards)
ts_rate = ts_cum_reward / n_pulls
print('Thompson sampling: total conversions', ts_cum_reward, 'rate', round(ts_rate, 4))
print('Bandit advantage (conversions):', ts_cum_reward - fixed_cum_reward)

The bandit earns more during the run by shifting traffic toward the better arm. That is what it buys.

**The cost.** Estimate each arm's rate with a 95% CI. Under the bandit, the losing arm gets few pulls - its CI blows up.

In [ ]:
def arm_ci(choices, rewards, arm):
    mask = np.array(choices) == arm
    n = mask.sum()
    if n == 0:
        return np.nan, np.nan, 0
    p_hat = np.array(rewards)[mask].mean()
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat - 1.96 * se, p_hat + 1.96 * se, n

fixed_lo, fixed_hi, fixed_n = arm_ci(choices, rewards, 0)
ts_lo, ts_hi, ts_n = arm_ci(ts_choices, ts_rewards, 0)
fixed_width = fixed_hi - fixed_lo
ts_width = ts_hi - ts_lo
print('Control arm pulls - fixed:', fixed_n, 'bandit:', ts_n)
print('CI width on losing arm - fixed:', round(fixed_width, 4), 'bandit:', round(ts_width, 4))

The bandit stopped sampling the losing arm, so you cannot precisely estimate its rate - causal clarity on that arm is what the bandit spent.

Plot cumulative reward over time for both policies.

In [ ]:
fig, ax = plt.subplots()
ax.plot(np.cumsum(rewards), label='fixed 50/50')
ax.plot(np.cumsum(ts_rewards), label='Thompson sampling')
ax.set_xlabel('pull')
ax.set_ylabel('cumulative conversions')
ax.set_title('Bandit earns more during the run')
ax.legend()
plt.show()

Higher cumulative reward is real - but it came from uneven sampling, not from magic.

## 6. What you do about it

- Use a **bandit** when the goal is maximising reward during learning (`U10-A13`).
- Use a **fixed split** when you need precise estimates on every arm.
- Report CI width on low-traffic arms before you claim you "know" a loser.

**When this matters less:** A clear winner with no downstream decision on the losing variant.

---

**Takeaway:** A bandit buys revenue during the experiment and spends it in wide confidence intervals on the arms it abandoned.

**Back to the unit:** [V2 unit 10](../V2/units/unit-10-analysis-decision-and-ethics/README.md)